In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import BallTree
from pyproj import Transformer

In [2]:
df = pd.read_csv("../DATA/PROCESS/target_dataset.csv")
df

,Unnamed: 0,datetime,자치구코드,위도,경도,hour,weekday,month,hour_sin,hour_cos,weekday_sin,weekday_cos,month_sin,month_cos,gu_walk_area,target
0,0,2024-12-29 23:54:00,11470.0,37.532463,126.833076,23,6,12,-0.258819,0.965926,-0.781831,0.623490,-2.449294e-16,1.000000,328170.6,0
1,1,2024-12-30 00:01:00,11470.0,37.532463,126.833076,0,0,12,0.000000,1.000000,0.000000,1.000000,-2.449294e-16,1.000000,328170.6,0
2,2,2024-12-30 00:11:00,11470.0,37.532463,126.833076,0,0,12,0.000000,1.000000,0.000000,1.000000,-2.449294e-16,1.000000,328170.6,0
3,3,2024-12-30 00:21:00,11470.0,37.532463,126.833076,0,0,12,0.000000,1.000000,0.000000,1.000000,-2.449294e-16,1.000000,328170.6,0
4,4,2024-12-30 00:31:00,11470.0,37.532463,126.833076,0,0,12,0.000000,1.000000,0.000000,1.000000,-2.449294e-16,1.000000,328170.6,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4033901,4033901,2025-11-28 13:40:00,11215.0,37.548059,127.106843,13,4,11,-0.258819,-0.965926,-0.433884,-0.900969,-5.000000e-01,0.866025,213179.6,1
4033902,4033902,2025-11-28 13:50:00,11215.0,37.548059,127.106843,13,4,11,-0.258819,-0.965926,-0.433884,-0.900969,-5.000000e-01,0.866025,213179.6,1
4033903,4033903,2025-11-28 14:00:00,11215.0,37.548059,127.106843,14,4,11,-0.500000,-0.866025,-0.433884,-0.900969,-5.000000e-01,0.866025,213179.6,1
4033904,4033904,2025-11-28 14:10:00,11215.0,37.548059,127.106843,14,4,11,-0.500000,-0.866025,-0.433884,-0.900969,-5.000000e-01,0.866025,213179.6,1


In [3]:
gu_code_map = {
    "종로구": 11110, "중구": 11140, "용산구": 11170, "성동구": 11200,
    "광진구": 11215, "동대문구": 11230, "중랑구": 11260, "성북구": 11290,
    "강북구": 11305, "도봉구": 11320, "노원구": 11350, "은평구": 11380,
    "서대문구": 11410, "마포구": 11440, "양천구": 11470, "강서구": 11500,
    "구로구": 11530, "금천구": 11545, "영등포구": 11560, "동작구": 11590,
    "관악구": 11620, "서초구": 11650, "강남구": 11680, "송파구": 11710,
    "강동구": 11740,
}

## **CULTURE**

In [4]:
concert = pd.read_csv("../DATA/FEATURE/CULTURE/concert.csv")
culture = pd.read_csv("../DATA/FEATURE/CULTURE/culture.csv")
movie = pd.read_csv("../DATA/FEATURE/CULTURE/movie.csv")

In [5]:
print(concert.columns)
concert.head(100)

Index(['자치구별(1)', '자치구별(2)', '2024', '2024.1', '2024.2', '2024.3', '2024.4',
       '2024.5'],
      dtype='str')


,자치구별(1),자치구별(2),2024,2024.1,2024.2,2024.3,2024.4,2024.5
0,자치구별(1),자치구별(2),계,운영주체별,운영주체별,규모별,규모별,규모별
1,자치구별(1),자치구별(2),소계,공공공연장,민간공연장,대공연장(1000석 이상),일반공연장(300~999석),소공연장(300석 미만)
2,합계,소계,445,109,336,20,101,324
3,합계,종로구,161,15,146,1,22,138
4,합계,중구,29,13,16,2,16,11
5,합계,용산구,12,5,7,2,4,6
6,합계,성동구,8,3,5,-,2,6
7,합계,광진구,16,4,12,2,6,8
8,합계,동대문구,3,-,3,-,-,3
9,합계,중랑구,1,1,-,-,1,-


In [6]:
# 불필요한 상단 2행 제거
concert = concert.iloc[2:].reset_index(drop=True)

# 컬럼 이름 재정의
concert.columns = [
    "level1", "gu_name", "total",
    "public", "private",
    "large_hall", "mid_hall", "small_hall"
]

# "합계" 행 제거
concert = concert[concert["gu_name"] != "소계"]

# '-' → 0 처리
concert = concert.replace("-", 0)

# 숫자형 변환
num_cols = ["total", "public", "private", "large_hall", "mid_hall", "small_hall"]
concert[num_cols] = concert[num_cols].astype(int)

In [7]:
concert["자치구코드"] = concert["gu_name"].map(gu_code_map)

In [8]:
df = df.merge(
    concert[[
        "자치구코드",
        "total", "public", "private",
        "large_hall", "mid_hall", "small_hall"
    ]],
    on="자치구코드",
    how="left"
)

In [9]:
df.rename(columns={
    "total": "concert_total",
    "public": "concert_public",
    "private": "concert_private",
    "large_hall": "concert_large",
    "mid_hall": "concert_mid",
    "small_hall": "concert_small",
}, inplace=True)

In [10]:
print(df.shape)
df.head()

(4033906, 22)


,Unnamed: 0,datetime,자치구코드,위도,경도,hour,weekday,month,hour_sin,hour_cos,...,month_sin,month_cos,gu_walk_area,target,concert_total,concert_public,concert_private,concert_large,concert_mid,concert_small
0,0,2024-12-29 23:54:00,11470.0,37.532463,126.833076,23,6,12,-0.258819,0.965926,...,-2.449294e-16,1.0,328170.6,0,6,4,2,0,3,3
1,1,2024-12-30 00:01:00,11470.0,37.532463,126.833076,0,0,12,0.000000,1.000000,...,-2.449294e-16,1.0,328170.6,0,6,4,2,0,3,3
2,2,2024-12-30 00:11:00,11470.0,37.532463,126.833076,0,0,12,0.000000,1.000000,...,-2.449294e-16,1.0,328170.6,0,6,4,2,0,3,3
3,3,2024-12-30 00:21:00,11470.0,37.532463,126.833076,0,0,12,0.000000,1.000000,...,-2.449294e-16,1.0,328170.6,0,6,4,2,0,3,3
4,4,2024-12-30 00:31:00,11470.0,37.532463,126.833076,0,0,12,0.000000,1.000000,...,-2.449294e-16,1.0,328170.6,0,6,4,2,0,3,3


In [11]:
print(culture.columns)
culture.head(100)

Index(['자치구별(1)', '자치구별(2)', '2022', '2022.1', '2022.2', '2022.3', '2022.4',
       '2022.5', '2022.6', '2022.7', '2022.8', '2022.9', '2022.10', '2022.11',
       '2022.12', '2022.13', '2022.14', '2022.15', '2022.16', '2022.17'],
      dtype='str')


,자치구별(1),자치구별(2),2022,2022.1,2022.2,2022.3,2022.4,2022.5,2022.6,2022.7,2022.8,2022.9,2022.10,2022.11,2022.12,2022.13,2022.14,2022.15,2022.16,2022.17
0,자치구별(1),자치구별(2),문화재,문화재,문화재,문화재,문화재,문화재,문화재,문화재,문화재,문화재,문화재,문화재,문화재,문화재,문화재,문화재,문화재,문화재
1,자치구별(1),자치구별(2),소계,지정문화재,지정문화재,지정문화재,지정문화재,지정문화재,지정문화재,지정문화재,지정문화재,지정문화재,지정문화재,지정문화재,지정문화재,지정문화재,지정문화재,지정문화재,등록문화재,등록문화재
2,자치구별(1),자치구별(2),소계,국가지정문화재,국가지정문화재,국가지정문화재,국가지정문화재,국가지정문화재,국가지정문화재,국가지정문화재,국가지정문화재,국가지정문화재,시지정문화재,시지정문화재,시지정문화재,시지정문화재,시지정문화재,문화재자료,국가등록문화재,시등록문화재
3,자치구별(1),자치구별(2),소계,소계,국보,보물,사적,명승,사적 및 명승,천연기념물,민속문화재,중요무형문화재,소계,유형문화재,기념물,민속문화재,무형문화재,소계,소계,소계
4,합계,소계,2043,1084,168,758,69,3,-,12,42,32,627,497,40,35,55,81,233,18
5,합계,종로구,483,225,17,154,26,2,-,10,11,5,173,148,6,15,4,27,57,1
6,합계,중구,108,57,8,41,8,-,-,-,-,-,30,15,8,7,-,2,19,-
7,합계,용산구,447,403,96,297,3,-,-,-,7,-,13,11,1,1,-,-,24,7
8,합계,성동구,32,1,-,1,-,-,-,-,-,-,25,24,-,1,-,3,2,1
9,합계,광진구,30,21,1,2,2,-,-,-,16,-,6,3,1,1,1,-,3,-


In [12]:
# 상단 메타 헤더 제거
culture = culture.iloc[4:].reset_index(drop=True)

# 컬럼명 재정의
culture.columns = [
    "level1", "gu_name",
    "heritage_total",
    "designated_total",
    "national_treasure",
    "treasure",
    "historic_site",
    "scenic_site",
    "historic_and_scenic_site",
    "natural_monument",
    "folk_cultural_property",
    "intangible_cultural_property",
    "city_designated_total",
    "tangible_cultural_property",
    "city_monument",
    "city_folk_cultural_property",
    "city_intangible_cultural_property",
    "cultural_property_material",
    "national_registered_cultural_property",
    "city_registered_cultural_property"
]

# 합계/기타 제거
culture = culture[
    (culture["gu_name"] != "소계") &
    (culture["gu_name"] != "기타")
].copy()

# '-' → 0
culture = culture.replace("-", 0)

# 숫자형 변환
num_cols = culture.columns.drop(["level1", "gu_name"])
culture[num_cols] = culture[num_cols].astype(int)

# 자치구코드 매핑
culture["자치구코드"] = culture["gu_name"].map(gu_code_map)

culture.head()

,level1,gu_name,heritage_total,designated_total,national_treasure,treasure,historic_site,scenic_site,historic_and_scenic_site,natural_monument,...,intangible_cultural_property,city_designated_total,tangible_cultural_property,city_monument,city_folk_cultural_property,city_intangible_cultural_property,cultural_property_material,national_registered_cultural_property,city_registered_cultural_property,자치구코드
1,합계,종로구,483,225,17,154,26,2,0,10,...,5,173,148,6,15,4,27,57,1,11110
2,합계,중구,108,57,8,41,8,0,0,0,...,0,30,15,8,7,0,2,19,0,11140
3,합계,용산구,447,403,96,297,3,0,0,0,...,0,13,11,1,1,0,0,24,7,11170
4,합계,성동구,32,1,0,1,0,0,0,0,...,0,25,24,0,1,0,3,2,1,11200
5,합계,광진구,30,21,1,2,2,0,0,0,...,0,6,3,1,1,1,0,3,0,11215


In [13]:
df = df.merge(
    culture.drop(columns=["level1", "gu_name"]),
    on="자치구코드",
    how="left"
)

In [14]:
print(df.shape)
df.head()

(4033906, 40)


,Unnamed: 0,datetime,자치구코드,위도,경도,hour,weekday,month,hour_sin,hour_cos,...,folk_cultural_property,intangible_cultural_property,city_designated_total,tangible_cultural_property,city_monument,city_folk_cultural_property,city_intangible_cultural_property,cultural_property_material,national_registered_cultural_property,city_registered_cultural_property
0,0,2024-12-29 23:54:00,11470.0,37.532463,126.833076,23,6,12,-0.258819,0.965926,...,0,0,2,2,0,0,0,0,0,0
1,1,2024-12-30 00:01:00,11470.0,37.532463,126.833076,0,0,12,0.000000,1.000000,...,0,0,2,2,0,0,0,0,0,0
2,2,2024-12-30 00:11:00,11470.0,37.532463,126.833076,0,0,12,0.000000,1.000000,...,0,0,2,2,0,0,0,0,0,0
3,3,2024-12-30 00:21:00,11470.0,37.532463,126.833076,0,0,12,0.000000,1.000000,...,0,0,2,2,0,0,0,0,0,0
4,4,2024-12-30 00:31:00,11470.0,37.532463,126.833076,0,0,12,0.000000,1.000000,...,0,0,2,2,0,0,0,0,0,0


In [15]:
print(movie.columns)
movie.head(100)

Index(['자치구별(1)', '자치구별(2)', '2024', '2024.1', '2024.2'], dtype='str')


,자치구별(1),자치구별(2),2024,2024.1,2024.2
0,자치구별(1),자치구별(2),개소수 (개소),스크린수 (개),좌석수 (개)
1,합계,소계,95,587,90363
2,합계,종로구,6,21,3648
3,합계,중구,8,32,4188
4,합계,용산구,3,33,6250
5,합계,성동구,1,11,2333
6,합계,광진구,7,46,6548
7,합계,동대문구,1,8,1737
8,합계,중랑구,3,22,3052
9,합계,성북구,2,8,1209


In [16]:
# 첫 행 제거
movie = movie.iloc[1:].reset_index(drop=True)

# 컬럼명 재정의
movie.columns = [
    "level1", "gu_name",
    "movie_theater_cnt",
    "movie_screen_cnt",
    "movie_seat_cnt"
]

# 소계 제거
movie = movie[movie["gu_name"] != "소계"].copy()

In [17]:
# 숫자형 변환
num_cols = ["movie_theater_cnt", "movie_screen_cnt", "movie_seat_cnt"]
movie[num_cols] = movie[num_cols].astype(int)

movie["자치구코드"] = movie["gu_name"].map(gu_code_map)

In [18]:
df = df.merge(
    movie[[
        "자치구코드",
        "movie_theater_cnt",
        "movie_screen_cnt",
        "movie_seat_cnt"
    ]],
    on="자치구코드",
    how="left"
)

In [19]:
print(df.shape)
df.head()

(4033906, 43)


,Unnamed: 0,datetime,자치구코드,위도,경도,hour,weekday,month,hour_sin,hour_cos,...,tangible_cultural_property,city_monument,city_folk_cultural_property,city_intangible_cultural_property,cultural_property_material,national_registered_cultural_property,city_registered_cultural_property,movie_theater_cnt,movie_screen_cnt,movie_seat_cnt
0,0,2024-12-29 23:54:00,11470.0,37.532463,126.833076,23,6,12,-0.258819,0.965926,...,2,0,0,0,0,0,0,2,17,2467
1,1,2024-12-30 00:01:00,11470.0,37.532463,126.833076,0,0,12,0.000000,1.000000,...,2,0,0,0,0,0,0,2,17,2467
2,2,2024-12-30 00:11:00,11470.0,37.532463,126.833076,0,0,12,0.000000,1.000000,...,2,0,0,0,0,0,0,2,17,2467
3,3,2024-12-30 00:21:00,11470.0,37.532463,126.833076,0,0,12,0.000000,1.000000,...,2,0,0,0,0,0,0,2,17,2467
4,4,2024-12-30 00:31:00,11470.0,37.532463,126.833076,0,0,12,0.000000,1.000000,...,2,0,0,0,0,0,0,2,17,2467


In [20]:
df.to_csv("../DATA/PROCESS/save_dataset.csv", index=False)

## **SALES**

In [21]:
import chardet

with open("../DATA/FEATURE/SALES/FOOD1.csv", "rb") as f:
    print(chardet.detect(f.read(10000)))

{'encoding': 'cp1250', 'confidence': 0.34222844773432537, 'language': 'sr', 'mime_type': 'text/plain'}


In [22]:
food1 = pd.read_csv("../DATA/FEATURE/SALES/FOOD1.csv", encoding="cp1250")
food2 = pd.read_csv("../DATA/FEATURE/SALES/FOOD2.csv", encoding="cp949")
shop = pd.read_csv("../DATA/FEATURE/SALES/SHOP.csv", encoding="cp949")

/var/folders/g8/gwqmqng10_g3h7m8r02yq1qc0000gn/T/ipykernel_72484/720638667.py:1: DtypeWarning: Columns (0: ŔüČ­ąřČŁ, 1: °Çą°ĽŇŔŻ±¸şĐ¸í, 2: ŔüĹëľ÷ĽŇÁöÁ¤ąřČŁ, 3: ŔüĹëľ÷ĽŇÁÖµČŔ˝˝Ä, 4: Č¨ĆäŔĚÁö) have mixed types. Specify dtype option on import or set low_memory=False.
  food1 = pd.read_csv("../DATA/FEATURE/SALES/FOOD1.csv", encoding="cp1250")


In [23]:
food1 = pd.read_csv(
    "../DATA/FEATURE/SALES/FOOD1.csv",
    encoding="cp949",
    encoding_errors="replace",
    low_memory=False
)

파일 대부분은 cp949인데, 중간에 0x82 같은 cp949로 해석 불가능한 이상한 바이트가 섞여 있는 상황일 가능성이 큼.

encoding_errors="replace"를 붙이면 이상한 글자는 �로 대체하고 계속 읽음.

In [24]:
print(food1.columns)
food1.head(100)

Index(['개방자치단체코드', '관리번호', '인허가일자', '영업상태코드', '영업상태명', '상세영업상태코드', '상세영업상태명',
       '폐업일자', '전화번호', '소재지면적', '소재지우편번호', '지번주소', '도로명주소', '도로명우편번호', '사업장명',
       '최종수정일자', '데이터갱신구분', '데이터갱신일자', '업태구분명', '좌표정보(X)', '좌표정보(Y)', '위생업태명',
       '남성종사자수', '여성종사자수', '영업장주변구분명', '등급구분명', '급수시설구분명', '본사종업원수',
       '공장사무직종업원수', '공장판매직종업원수', '공장생산직종업원수', '건물소유구분명', '보증액', '월세액',
       '다중이용업소여부', '시설총규모', '전통업소지정번호', '전통업소주된음식', '홈페이지'],
      dtype='str')


,개방자치단체코드,관리번호,인허가일자,영업상태코드,영업상태명,상세영업상태코드,상세영업상태명,폐업일자,전화번호,소재지면적,...,공장판매직종업원수,공장생산직종업원수,건물소유구분명,보증액,월세액,다중이용업소여부,시설총규모,전통업소지정번호,전통업소주된음식,홈페이지
0,3020000,3020000-101-2001-07985,2001-05-23,3,폐업,2,폐업,2007-02-07,02-796-2255,29.70,...,NaN,NaN,NaN,NaN,NaN,N,29.70,NaN,NaN,NaN
1,3230000,3230000-101-2019-00201,2019-03-26,3,폐업,2,폐업,2021-03-31,NaN,28.00,...,NaN,NaN,NaN,NaN,NaN,N,28.00,NaN,NaN,NaN
2,3120000,3120000-101-1994-02043,1994-06-15,3,폐업,2,폐업,2002-06-17,02-0393-9083,43.20,...,NaN,NaN,NaN,NaN,NaN,N,43.20,NaN,NaN,NaN
3,3110000,3110000-101-2005-00128,2005-04-26,3,폐업,2,폐업,2013-05-29,02-306-2168,190.24,...,NaN,NaN,NaN,NaN,NaN,N,190.24,NaN,NaN,NaN
4,3000000,3000000-101-2007-00202,2007-08-27,1,영업/정상,1,영업,,02-739-3345,85.26,...,NaN,NaN,NaN,NaN,NaN,N,85.26,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,3160000,3160000-101-1984-00958,1984-02-13,3,폐업,2,폐업,1996-06-11,02-854-1814,25.84,...,NaN,NaN,NaN,NaN,NaN,N,25.84,NaN,NaN,NaN
96,3220000,3220000-101-1999-19709,1999-07-12,3,폐업,2,폐업,2003-09-22,02-557-3317,17.27,...,NaN,NaN,NaN,NaN,NaN,N,17.27,NaN,NaN,NaN
97,3010000,3010000-101-2009-00015,2009-01-28,1,영업/정상,1,영업,,02-755-8983,49.50,...,NaN,NaN,NaN,NaN,NaN,N,49.50,NaN,NaN,NaN
98,3220000,3220000-101-2001-23675,2001-06-19,3,폐업,2,폐업,2007-09-11,02-481-8239,72.72,...,NaN,NaN,NaN,NaN,NaN,N,72.72,NaN,NaN,NaN


In [25]:
# 1. 음식점 데이터 전처리

food = food1.copy()

food = food[food["영업상태명"] == "영업/정상"].copy()

for col in ["좌표정보(X)", "좌표정보(Y)", "소재지면적", "시설총규모"]:
    food[col] = pd.to_numeric(food[col], errors="coerce")

food = food.dropna(subset=["좌표정보(X)", "좌표정보(Y)"]).copy()

print(food[["좌표정보(X)", "좌표정보(Y)"]].describe())

             좌표정보(X)        좌표정보(Y)
count  107117.000000  107117.000000
mean   198998.842206  449199.452134
std      7128.032614    5265.926160
min    180943.793440  436477.114292
25%    193059.057913  444977.866294
50%    199872.928165  448974.693085
75%    204389.263696  452169.826735
max    215813.845000  465406.967121


In [26]:
# EPSG:5174 -> WGS84 위경도
transformer = Transformer.from_crs("EPSG:5174", "EPSG:4326", always_xy=True)

lon, lat = transformer.transform(
    food["좌표정보(X)"].values,
    food["좌표정보(Y)"].values
)

food["경도"] = lon
food["위도"] = lat

# 서울 범위 필터
food_local = food[
    (food["위도"].between(37.0, 38.0)) &
    (food["경도"].between(126.0, 128.0))
].copy()

print(food_local.shape)
print(food_local[["위도", "경도"]].head())

(107117, 41)
           위도          경도
4   37.573587  126.972094
6   37.536316  127.059877
7   37.581062  127.089234
14  37.535276  126.973176
35  37.575734  126.837836


In [27]:
food_coords = np.radians(
    food_local[["위도", "경도"]].values
)

sensor_coords = np.radians(
    df[["위도", "경도"]].values
)

tree = BallTree(food_coords, metric="haversine")

EARTH_RADIUS = 6371000

for r in [100, 300, 500, 1000]:
    radius = r / EARTH_RADIUS
    df[f"rest_cnt_{r}m"] = tree.query_radius(
        sensor_coords,
        r=radius,
        count_only=True
    )

dist, ind = tree.query(sensor_coords, k=1)
df["nearest_rest_dist_m"] = dist[:, 0] * EARTH_RADIUS

k = min(20, len(food_local))
dist_k, ind_k = tree.query(sensor_coords, k=k)

dist_m = dist_k * EARTH_RADIUS
df["rest_weighted_density_k20"] = (1 / (dist_m + 1)).sum(axis=1)
df["rest_mean_dist_k20_m"] = dist_m.mean(axis=1)

print(df[[
    "rest_cnt_100m",
    "rest_cnt_300m",
    "rest_cnt_500m",
    "rest_cnt_1000m",
    "nearest_rest_dist_m",
    "rest_weighted_density_k20"
]].head())

   rest_cnt_100m  rest_cnt_300m  rest_cnt_500m  rest_cnt_1000m  \
0             22            100            258             824   
1             22            100            258             824   
2             22            100            258             824   
3             22            100            258             824   
4             22            100            258             824   

   nearest_rest_dist_m  rest_weighted_density_k20  
0             7.487526                    0.58049  
1             7.487526                    0.58049  
2             7.487526                    0.58049  
3             7.487526                    0.58049  
4             7.487526                    0.58049  


In [28]:
df.to_csv("../DATA/PROCESS/save_dataset1.csv", index=False)

In [29]:
print(df.shape)
df.head()

(4033906, 50)


,Unnamed: 0,datetime,자치구코드,위도,경도,hour,weekday,month,hour_sin,hour_cos,...,movie_theater_cnt,movie_screen_cnt,movie_seat_cnt,rest_cnt_100m,rest_cnt_300m,rest_cnt_500m,rest_cnt_1000m,nearest_rest_dist_m,rest_weighted_density_k20,rest_mean_dist_k20_m
0,0,2024-12-29 23:54:00,11470.0,37.532463,126.833076,23,6,12,-0.258819,0.965926,...,2,17,2467,22,100,258,824,7.487526,0.58049,43.89043
1,1,2024-12-30 00:01:00,11470.0,37.532463,126.833076,0,0,12,0.000000,1.000000,...,2,17,2467,22,100,258,824,7.487526,0.58049,43.89043
2,2,2024-12-30 00:11:00,11470.0,37.532463,126.833076,0,0,12,0.000000,1.000000,...,2,17,2467,22,100,258,824,7.487526,0.58049,43.89043
3,3,2024-12-30 00:21:00,11470.0,37.532463,126.833076,0,0,12,0.000000,1.000000,...,2,17,2467,22,100,258,824,7.487526,0.58049,43.89043
4,4,2024-12-30 00:31:00,11470.0,37.532463,126.833076,0,0,12,0.000000,1.000000,...,2,17,2467,22,100,258,824,7.487526,0.58049,43.89043


In [30]:
print(food2.columns)
food1.head(100)

Index(['개방자치단체코드', '관리번호', '인허가일자', '인허가취소일자', '영업상태코드', '영업상태명', '상세영업상태코드',
       '상세영업상태명', '폐업일자', '휴업시작일자', '휴업종료일자', '전화번호', '소재지우편번호', '지번주소',
       '도로명주소', '도로명우편번호', '사업장명', '최종수정일자', '데이터갱신구분', '데이터갱신일자', '좌표정보(X)',
       '좌표정보(Y)', '문화체육업종명', '문화사업자구분명', '지역구분명', '총층수', '주변환경명', '제작취급품목내용',
       '보험기관명', '건물용도명', '지상층수', '지하층수', '객실수', '건축연면적', '영문상호명', '영문상호주소',
       '선박총톤수', '선박척수', '선박제원', '무대면적', '좌석수', '기념품종류', '회의실별동시수용인원', '시설면적',
       '자본금', '보험시작일자', '보험종료일자', '시설규모'],
      dtype='str')


,개방자치단체코드,관리번호,인허가일자,영업상태코드,영업상태명,상세영업상태코드,상세영업상태명,폐업일자,전화번호,소재지면적,...,공장판매직종업원수,공장생산직종업원수,건물소유구분명,보증액,월세액,다중이용업소여부,시설총규모,전통업소지정번호,전통업소주된음식,홈페이지
0,3020000,3020000-101-2001-07985,2001-05-23,3,폐업,2,폐업,2007-02-07,02-796-2255,29.70,...,NaN,NaN,NaN,NaN,NaN,N,29.70,NaN,NaN,NaN
1,3230000,3230000-101-2019-00201,2019-03-26,3,폐업,2,폐업,2021-03-31,NaN,28.00,...,NaN,NaN,NaN,NaN,NaN,N,28.00,NaN,NaN,NaN
2,3120000,3120000-101-1994-02043,1994-06-15,3,폐업,2,폐업,2002-06-17,02-0393-9083,43.20,...,NaN,NaN,NaN,NaN,NaN,N,43.20,NaN,NaN,NaN
3,3110000,3110000-101-2005-00128,2005-04-26,3,폐업,2,폐업,2013-05-29,02-306-2168,190.24,...,NaN,NaN,NaN,NaN,NaN,N,190.24,NaN,NaN,NaN
4,3000000,3000000-101-2007-00202,2007-08-27,1,영업/정상,1,영업,,02-739-3345,85.26,...,NaN,NaN,NaN,NaN,NaN,N,85.26,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,3160000,3160000-101-1984-00958,1984-02-13,3,폐업,2,폐업,1996-06-11,02-854-1814,25.84,...,NaN,NaN,NaN,NaN,NaN,N,25.84,NaN,NaN,NaN
96,3220000,3220000-101-1999-19709,1999-07-12,3,폐업,2,폐업,2003-09-22,02-557-3317,17.27,...,NaN,NaN,NaN,NaN,NaN,N,17.27,NaN,NaN,NaN
97,3010000,3010000-101-2009-00015,2009-01-28,1,영업/정상,1,영업,,02-755-8983,49.50,...,NaN,NaN,NaN,NaN,NaN,N,49.50,NaN,NaN,NaN
98,3220000,3220000-101-2001-23675,2001-06-19,3,폐업,2,폐업,2007-09-11,02-481-8239,72.72,...,NaN,NaN,NaN,NaN,NaN,N,72.72,NaN,NaN,NaN


In [31]:
# 동일 사업장 여부 확인
len(set(food1["관리번호"]) & set(food2["관리번호"]))

0

In [32]:
EARTH_RADIUS = 6371000

# 1. food2 전처리

tour = food2.copy()

# 영업 중만 사용
tour = tour[tour["영업상태명"] == "영업/정상"].copy()

for col in ["좌표정보(X)", "좌표정보(Y)"]:
    tour[col] = pd.to_numeric(tour[col], errors="coerce")

tour = tour.dropna(subset=["좌표정보(X)", "좌표정보(Y)"]).copy()

# 2. 좌표 변환 후보 테스트

candidate_epsg = ["EPSG:5174", "EPSG:5181", "EPSG:5186", "EPSG:5187"]

best_epsg = None
best_valid = -1

for epsg in candidate_epsg:
    transformer = Transformer.from_crs(epsg, "EPSG:4326", always_xy=True)

    lon, lat = transformer.transform(
        tour["좌표정보(X)"].values,
        tour["좌표정보(Y)"].values
    )

    valid = (
        pd.Series(lat).between(37.0, 38.0) &
        pd.Series(lon).between(126.0, 128.0)
    ).sum()

    print(epsg, "서울 범위 좌표 수:", valid)

    if valid > best_valid:
        best_valid = valid
        best_epsg = epsg

print("선택된 EPSG:", best_epsg)


# 3. 선택된 좌표계로 변환

transformer = Transformer.from_crs(best_epsg, "EPSG:4326", always_xy=True)

lon, lat = transformer.transform(
    tour["좌표정보(X)"].values,
    tour["좌표정보(Y)"].values
)

tour["경도"] = lon
tour["위도"] = lat

tour_local = tour[
    (tour["위도"].between(37.0, 38.0)) &
    (tour["경도"].between(126.0, 128.0))
].copy()

print("사용 가능한 관광업소 좌표 수:", len(tour_local))


# 4. food2 기반 로컬 피처 생성


tour_coords = np.radians(
    tour_local[["위도", "경도"]].values
)

sensor_coords = np.radians(
    df[["위도", "경도"]].values
)

tree = BallTree(tour_coords, metric="haversine")

# 반경별 관광업소 개수
for r in [100, 300, 500, 1000]:
    radius = r / EARTH_RADIUS

    df[f"tour_food_cnt_{r}m"] = tree.query_radius(
        sensor_coords,
        r=radius,
        count_only=True
    )

# 가장 가까운 관광업소 거리
dist, ind = tree.query(sensor_coords, k=1)
df["nearest_tour_food_dist_m"] = dist[:, 0] * EARTH_RADIUS

# 거리 가중 관광업소 밀도
k = min(20, len(tour_local))
dist_k, ind_k = tree.query(sensor_coords, k=k)

dist_m = dist_k * EARTH_RADIUS

df["tour_food_weighted_density_k20"] = (1 / (dist_m + 1)).sum(axis=1)
df["tour_food_mean_dist_k20_m"] = dist_m.mean(axis=1)

# 5. 기존 food1 피처와 비율 생성


if "rest_cnt_300m" in df.columns:
    df["tour_food_ratio_300m"] = (
        df["tour_food_cnt_300m"] / (df["rest_cnt_300m"] + 1)
    )

if "rest_cnt_500m" in df.columns:
    df["tour_food_ratio_500m"] = (
        df["tour_food_cnt_500m"] / (df["rest_cnt_500m"] + 1)
    )


# 6. check

tour_food_feature_cols = [
    col for col in df.columns
    if col.startswith("tour_food_") or col.startswith("nearest_tour_food")
]

print(df[tour_food_feature_cols].head())
print(df[tour_food_feature_cols].describe())

EPSG:5174 서울 범위 좌표 수: 122
EPSG:5181 서울 범위 좌표 수: 122
EPSG:5186 서울 범위 좌표 수: 0
EPSG:5187 서울 범위 좌표 수: 0
선택된 EPSG: EPSG:5174
사용 가능한 관광업소 좌표 수: 122
   tour_food_cnt_100m  tour_food_cnt_300m  tour_food_cnt_500m  \
0                   0                   0                   0   
1                   0                   0                   0   
2                   0                   0                   0   
3                   0                   0                   0   
4                   0                   0                   0   

   tour_food_cnt_1000m  nearest_tour_food_dist_m  \
0                    0                1050.83364   
1                    0                1050.83364   
2                    0                1050.83364   
3                    0                1050.83364   
4                    0                1050.83364   

   tour_food_weighted_density_k20  tour_food_mean_dist_k20_m  \
0                        0.003776                7297.574006   
1                        0

In [33]:
print(shop.columns)
shop.head(100)

Index(['개방자치단체코드', '관리번호', '인허가일자', '인허가취소일자', '영업상태코드', '영업상태명', '상세영업상태코드',
       '상세영업상태명', '폐업일자', '휴업시작일자', '휴업종료일자', '재개업일자', '전화번호', '소재지면적',
       '소재지우편번호', '지번주소', '도로명주소', '도로명우편번호', '사업장명', '최종수정일자', '데이터갱신구분',
       '데이터갱신일자', '업태구분명', '좌표정보(X)', '좌표정보(Y)', '점포구분명'],
      dtype='str')


,개방자치단체코드,관리번호,인허가일자,인허가취소일자,영업상태코드,영업상태명,상세영업상태코드,상세영업상태명,폐업일자,휴업시작일자,...,도로명주소,도로명우편번호,사업장명,최종수정일자,데이터갱신구분,데이터갱신일자,업태구분명,좌표정보(X),좌표정보(Y),점포구분명
0,3050000,2018305014007500001,2018-01-25,,1,영업/정상,1,정상영업,,,...,"서울특별시 동대문구 고산자로36길 3, 2층 (제기동)",02571,노브랜드 경동시장점,2024-04-30 16:41:21,I,2025-12-15 16:15:28,구분없음,203403.019535366,452985.725419514,준대규모점포
1,3050000,2022305014007500001,2022-07-11,,1,영업/정상,1,정상영업,,,...,"서울특별시 동대문구 고산자로32길 78 (용두동, 청량리역한양수자인그라시엘)",02561,청량리역 한양수자인 그라시엘 판매시설,2023-06-29 10:04:00,I,2025-12-15 16:15:28,그 밖의 대규모점포,203728.022004023,452778.965235022,대규모점포
2,3100000,1996310018407500001,1996-09-18,,1,영업/정상,1,정상영업,,,...,"서울특별시 노원구 한글비석로 232 (중계동, 유경데파트)",01734,유경데파트,2021-12-31 09:18:45,I,2025-12-15 16:15:28,그 밖의 대규모점포,206719.586259379,460669.836412326,대규모점포
3,3100000,1997310018407500001,1997-11-20,,1,영업/정상,1,정상영업,,,...,"서울특별시 노원구 동일로180길 14 (공릉동, 공릉종합상가)",,공릉종합상가,2022-01-14 14:38:53,I,2025-12-15 16:15:28,그 밖의 대규모점포,206578.976039388,457736.364674901,대규모점포
4,3100000,1998310009607500001,1998-12-31,,1,영업/정상,1,정상영업,,,...,서울특별시 노원구 한글비석로 396 (상계동),,상계역전종합상가,2026-04-27 13:37:32,U,2026-04-28 03:03:03,쇼핑센터,206280.789903642,462002.498987713,대규모점포
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,3080000,2009308009207500001,2009-09-10,,1,영업/정상,1,정상영업,,,...,서울특별시 강북구 도봉로 34 (미아동),,트레지오,2022-01-18 13:00:14,I,2026-04-30 22:35:17,복합쇼핑몰,202642.446045912,456636.308192293,대규모점포
96,3080000,2011308013907500001,2011-01-03,,1,영업/정상,1,정상영업,,,...,서울특별시 강북구 한천로148길 12-43 (수유동),,강북종합시장,2011-12-19 13:41:27,I,2026-04-30 22:35:18,그 밖의 대규모점포,,,대규모점포
97,3080000,2011308013907500002,2011-01-24,,1,영업/정상,1,정상영업,,,...,서울특별시 강북구 한천로123길 26 (번동),,강북 북부시장,2024-03-20 21:36:36,I,2026-04-30 22:35:18,그 밖의 대규모점포,202608.934103734,459320.030764534,대규모점포
98,3080000,2011308013907500004,2011-08-23,,1,영업/정상,1,정상영업,,,...,서울특별시 강북구 삼양로27길 35-21 (미아동),,㈜지에스리테일 GS수퍼 강북미아점,2025-05-23 12:05:15,I,2026-04-30 22:35:18,구분없음,201602.577531291,457290.100695375,준대규모점포


In [ ]:
from tqdm import tqdm
tqdm.pandas()  # optional

# 1) 센서 좌표 unique
sensor_unique = (
    df[["위도", "경도"]]
    .drop_duplicates()
    .reset_index(drop=True)
    .copy()
)

sensor_coords = np.radians(sensor_unique[["위도", "경도"]].values)
shop_coords = np.radians(shop_local[["위도", "경도"]].values)

tree = BallTree(shop_coords, metric="haversine")

# 2) 반경별 개수 (진행률 표시)
for r in tqdm([100, 300, 500, 1000], desc="반경별 shop count"):
    radius = r / EARTH_RADIUS
    sensor_unique[f"shop_cnt_{r}m"] = tree.query_radius(
        sensor_coords,
        r=radius,
        count_only=True
    )

# 3) nearest 거리
print("nearest 거리 계산 중...")
dist, ind = tree.query(sensor_coords, k=1)
sensor_unique["nearest_shop_dist_m"] = dist[:, 0] * EARTH_RADIUS

# 4) kNN 거리 기반
print("kNN density 계산 중...")
k = min(20, len(shop_local))
dist_k, ind_k = tree.query(sensor_coords, k=k)
dist_m = dist_k * EARTH_RADIUS

sensor_unique["shop_weighted_density_k20"] = (1 / (dist_m + 1)).sum(axis=1)
sensor_unique["shop_mean_dist_k20_m"] = dist_m.mean(axis=1)

# 5) 500m 내 다양성 (핵심 병목 → tqdm 필수)
radius_500 = 500 / EARTH_RADIUS
indices_500 = tree.query_radius(sensor_coords, r=radius_500)

def local_nunique(indices, col):
    if len(indices) == 0:
        return 0
    return shop_local.iloc[indices][col].dropna().nunique()

def local_entropy(indices, col):
    if len(indices) == 0:
        return 0
    values = shop_local.iloc[indices][col].dropna()
    if len(values) == 0:
        return 0
    p = values.value_counts(normalize=True)
    return -(p * np.log(p + 1e-9)).sum()

print("500m 다양성 계산 중...")

sensor_unique["shop_type_cnt_500m"] = [
    local_nunique(idx, "업태구분명")
    for idx in tqdm(indices_500, desc="type count")
]

sensor_unique["shop_type_entropy_500m"] = [
    local_entropy(idx, "업태구분명")
    for idx in tqdm(indices_500, desc="type entropy")
]

sensor_unique["shop_store_type_cnt_500m"] = [
    local_nunique(idx, "점포구분명")
    for idx in tqdm(indices_500, desc="store type count")
]

sensor_unique["shop_store_type_entropy_500m"] = [
    local_entropy(idx, "점포구분명")
    for idx in tqdm(indices_500, desc="store type entropy")
]

# 6) merge
print("merge 중...")
df = df.merge(
    sensor_unique,
    on=["위도", "경도"],
    how="left"
)

print("완료")

반경별 shop count: 100%|██████████| 4/4 [00:00<00:00, 800.02it/s]


nearest 거리 계산 중...
kNN density 계산 중...
500m 다양성 계산 중...


store type entropy: 100%|██████████| 97/97 [00:00<00:00, 1918.21it/s]


merge 중...
완료


In [38]:
df.head(100)

,Unnamed: 0,datetime,자치구코드,위도,경도,hour,weekday,month,hour_sin,hour_cos,...,shop_cnt_300m_y,shop_cnt_500m_y,shop_cnt_1000m_y,nearest_shop_dist_m_y,shop_weighted_density_k20_y,shop_mean_dist_k20_m_y,shop_type_cnt_500m_y,shop_type_entropy_500m,shop_store_type_cnt_500m,shop_store_type_entropy_500m
0,0,2024-12-29 23:54:00,11470.0,37.532463,126.833076,23,6,12,-0.258819,0.965926,...,0,0,3,766.951014,0.012089,2057.337914,0,0.0,0,0.0
1,1,2024-12-30 00:01:00,11470.0,37.532463,126.833076,0,0,12,0.000000,1.000000,...,0,0,3,766.951014,0.012089,2057.337914,0,0.0,0,0.0
2,2,2024-12-30 00:11:00,11470.0,37.532463,126.833076,0,0,12,0.000000,1.000000,...,0,0,3,766.951014,0.012089,2057.337914,0,0.0,0,0.0
3,3,2024-12-30 00:21:00,11470.0,37.532463,126.833076,0,0,12,0.000000,1.000000,...,0,0,3,766.951014,0.012089,2057.337914,0,0.0,0,0.0
4,4,2024-12-30 00:31:00,11470.0,37.532463,126.833076,0,0,12,0.000000,1.000000,...,0,0,3,766.951014,0.012089,2057.337914,0,0.0,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,2024-12-30 15:43:00,11470.0,37.532463,126.833076,15,0,12,-0.707107,-0.707107,...,0,0,3,766.951014,0.012089,2057.337914,0,0.0,0,0.0
96,96,2024-12-30 16:01:00,11470.0,37.532463,126.833076,16,0,12,-0.866025,-0.500000,...,0,0,3,766.951014,0.012089,2057.337914,0,0.0,0,0.0
97,97,2024-12-30 16:10:00,11470.0,37.532463,126.833076,16,0,12,-0.866025,-0.500000,...,0,0,3,766.951014,0.012089,2057.337914,0,0.0,0,0.0
98,98,2024-12-30 16:20:00,11470.0,37.532463,126.833076,16,0,12,-0.866025,-0.500000,...,0,0,3,766.951014,0.012089,2057.337914,0,0.0,0,0.0


In [ ]:
# main_shop_features = [
#     "shop_cnt_300m",
#     "shop_cnt_500m",
#     "shop_cnt_1000m",
#     "nearest_shop_dist_m",
#     "shop_weighted_density_k20",
#     "shop_type_entropy_500m",
#     "shop_density_by_walk_area",
# ]

In [ ]:
df.to_csv("../DATA/PROCESS/save_dataset2.csv")

## **TRAFFIC**

In [39]:
with open("../DATA/FEATURE/TRAFFIC/BUS.csv", "rb") as f:
    print(chardet.detect(f.read(10000)))

{'encoding': 'cp1250', 'confidence': 0.08196421029197, 'language': 'sr', 'mime_type': 'text/plain'}


In [40]:
bus = pd.read_csv("../DATA/FEATURE/TRAFFIC/BUS.csv", encoding="euc-kr")
subway = pd.read_csv("../DATA/FEATURE/TRAFFIC/SUBWAY.csv", encoding="euc-kr")

In [41]:
print(bus.shape)
bus.head()

(11480, 7)


,정류장_ID,정류장_명칭,정류장_유형,정류장_번호,위도,경도,버스도착정보안내기_설치_여부
0,100000001,종로2가사거리,중앙차로,1001,37.569806,126.987752,설치
1,100000002,창경궁.서울대학교병원,중앙차로,1002,37.579433,126.996521,설치
2,100000003,명륜3가.성대입구,중앙차로,1003,37.582580,126.998251,설치
3,100000004,종로2가.삼일교,중앙차로,1004,37.568579,126.987613,설치
4,100000005,혜화동로터리.여운형활동터,중앙차로,1005,37.586243,127.001744,설치


In [42]:
bus["has_info"] = (bus["버스도착정보안내기_설치_여부"] == "설치").astype(int)

In [43]:
# 위도/경도 → radians
bus_coords = np.radians(bus[["위도", "경도"]].values)
df_coords = np.radians(df[["위도", "경도"]].values)

# BallTree 생성
tree = BallTree(bus_coords, metric='haversine')

# 반경 설정 (예: 300m)
radius = 300 / 6371000  # meters → radians

# 각 센서 기준 반경 내 정류장 index
indices = tree.query_radius(df_coords, r=radius)

In [44]:
# 정류장 개수
df["bus_stop_cnt_300m"] = [len(idx) for idx in indices]

# 안내기 있는 정류장 수
bus_has_info = bus["has_info"].values

df["bus_info_cnt_300m"] = [
    bus_has_info[idx].sum() if len(idx) > 0 else 0
    for idx in indices
]

In [45]:
for r in [100, 300, 500]:
    rad = r / 6371000
    idxs = tree.query_radius(df_coords, r=rad)

    df[f"bus_stop_cnt_{r}m"] = [len(i) for i in idxs]

In [46]:
df.shape

(4033906, 82)

In [47]:
print(subway.shape)
subway.head()

(783, 5)


,역사_ID,역사명,호선,위도,경도
0,9010,동탄,수도권 광역급행철도,37.20034,127.09569
1,9009,구성,수도권 광역급행철도,37.29913,127.10389
2,9008,성남,수도권 광역급행철도,37.39467,127.12058
3,9007,수서,수도권 광역급행철도,37.48637,127.10161
4,9006,삼성,수도권 광역급행철도,37.50887,127.06324


In [49]:
# 1. subway 전처리
subway_local = subway.copy()

for col in ["위도", "경도"]:
    subway_local[col] = pd.to_numeric(subway_local[col], errors="coerce")

subway_local = subway_local.dropna(subset=["위도", "경도"]).copy()

subway_local = subway_local[
    (subway_local["위도"].between(37.0, 38.0)) &
    (subway_local["경도"].between(126.0, 128.0))
].copy()

print("사용 가능한 지하철역 수:", len(subway_local))

# 2. 센서 좌표 unique로 축소
sensor_unique = (
    df[["위도", "경도"]]
    .drop_duplicates()
    .reset_index(drop=True)
    .copy()
)

subway_coords = np.radians(subway_local[["위도", "경도"]].values)
sensor_coords = np.radians(sensor_unique[["위도", "경도"]].values)

tree = BallTree(subway_coords, metric="haversine")

# 3. 반경별 지하철역 개수
for r in [300, 500, 1000, 1500]:
    radius = r / EARTH_RADIUS

    sensor_unique[f"subway_cnt_{r}m"] = tree.query_radius(
        sensor_coords,
        r=radius,
        count_only=True
    )

# 4. 가장 가까운 지하철역 거리
dist, ind = tree.query(sensor_coords, k=1)
sensor_unique["nearest_subway_dist_m"] = dist[:, 0] * EARTH_RADIUS

nearest_idx = ind[:, 0]
sensor_unique["nearest_subway_line"] = (
    subway_local.iloc[nearest_idx]["호선"].values
)

# 5. 거리 가중 지하철 접근성
k = min(5, len(subway_local))
dist_k, ind_k = tree.query(sensor_coords, k=k)

dist_m = dist_k * EARTH_RADIUS

sensor_unique["subway_weighted_access_k5"] = (1 / (dist_m + 1)).sum(axis=1)
sensor_unique["subway_mean_dist_k5_m"] = dist_m.mean(axis=1)

# 6. 1km 내 호선 다양성
radius_1000 = 1000 / EARTH_RADIUS
indices_1000 = tree.query_radius(sensor_coords, r=radius_1000)

def line_count(indices):
    if len(indices) == 0:
        return 0
    return subway_local.iloc[indices]["호선"].dropna().nunique()

def line_entropy(indices):
    if len(indices) == 0:
        return 0
    values = subway_local.iloc[indices]["호선"].dropna()
    if len(values) == 0:
        return 0
    p = values.value_counts(normalize=True)
    return -(p * np.log(p + 1e-9)).sum()

sensor_unique["subway_line_cnt_1000m"] = [
    line_count(idx) for idx in indices_1000
]

sensor_unique["subway_line_entropy_1000m"] = [
    line_entropy(idx) for idx in indices_1000
]

# 7. 원본 df에 merge
df = df.merge(
    sensor_unique,
    on=["위도", "경도"],
    how="left"
)

# 8. 결측 처리
subway_feature_cols = [
    col for col in df.columns
    if col.startswith("subway_") or col.startswith("nearest_subway")
]

df[subway_feature_cols] = df[subway_feature_cols].fillna(0)

print(df[subway_feature_cols].head())
print(df[subway_feature_cols].describe())

사용 가능한 지하철역 수: 769
   subway_cnt_300m_x  subway_cnt_500m_x  subway_cnt_1000m_x  \
0                  0                  0                   0   
1                  0                  0                   0   
2                  0                  0                   0   
3                  0                  0                   0   
4                  0                  0                   0   

   subway_cnt_1500m_x  nearest_subway_dist_m_x nearest_subway_line_x  \
0                   2              1198.634247                   5호선   
1                   2              1198.634247                   5호선   
2                   2              1198.634247                   5호선   
3                   2              1198.634247                   5호선   
4                   2              1198.634247                   5호선   

   subway_weighted_access_k5_x  subway_mean_dist_k5_m_x  subway_cnt_300m_y  \
0                     0.003112              1732.764524                  0   
1            

In [50]:
print(df.shape)
df.head()

(4033906, 100)


,Unnamed: 0,datetime,자치구코드,위도,경도,hour,weekday,month,hour_sin,hour_cos,...,subway_cnt_300m_y,subway_cnt_500m_y,subway_cnt_1000m_y,subway_cnt_1500m_y,nearest_subway_dist_m_y,nearest_subway_line_y,subway_weighted_access_k5_y,subway_mean_dist_k5_m_y,subway_line_cnt_1000m,subway_line_entropy_1000m
0,0,2024-12-29 23:54:00,11470.0,37.532463,126.833076,23,6,12,-0.258819,0.965926,...,0,0,0,2,1198.634247,5호선,0.003112,1732.764524,0,0.0
1,1,2024-12-30 00:01:00,11470.0,37.532463,126.833076,0,0,12,0.000000,1.000000,...,0,0,0,2,1198.634247,5호선,0.003112,1732.764524,0,0.0
2,2,2024-12-30 00:11:00,11470.0,37.532463,126.833076,0,0,12,0.000000,1.000000,...,0,0,0,2,1198.634247,5호선,0.003112,1732.764524,0,0.0
3,3,2024-12-30 00:21:00,11470.0,37.532463,126.833076,0,0,12,0.000000,1.000000,...,0,0,0,2,1198.634247,5호선,0.003112,1732.764524,0,0.0
4,4,2024-12-30 00:31:00,11470.0,37.532463,126.833076,0,0,12,0.000000,1.000000,...,0,0,0,2,1198.634247,5호선,0.003112,1732.764524,0,0.0


In [51]:
df.to_csv("../DATA/PROCESS/save_dataset3.csv")